# Diabetes Prediction Using Machine Learning Algorithms and Hyperparameter Tuning

This notebook recreates the project using the Pima Indians Diabetes dataset. It includes data loading, preprocessing, model training, evaluation, hyperparameter tuning, and single-person prediction.

**Target column:** `Outcome` where `0 = Non-Diabetic`, `1 = Diabetic`.

## 1. Install / import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

## 2. Load dataset

Option A works directly in Google Colab using a public CSV mirror. Option B lets you upload `diabetes.csv` manually if the link does not work.

In [ ]:
# Option A: Load from public URL
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url)

# Option B: If you downloaded diabetes.csv, comment Option A above and use:
# df = pd.read_csv("diabetes.csv")

df.head()

In [ ]:
print(df.shape)
print(df.columns.tolist())
df.info()

In [ ]:
df.describe()

In [ ]:
df['Outcome'].value_counts().plot(kind='pie', autopct='%1.1f%%', title='Diabetes Outcome')
plt.ylabel('')
plt.show()

print("0 = Non-Diabetic")
print("1 = Diabetic")

## 3. Preprocessing

In this dataset, some medical values are recorded as `0`, which is not realistic for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. We replace those zeros with missing values and then fill them using the column mean.

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

cols_with_zero_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

X[cols_with_zero_missing] = X[cols_with_zero_missing].replace(0, np.nan)
X = X.fillna(X.mean())

X.describe()

In [ ]:
X.boxplot(figsize=(14, 6))
plt.title('Feature Distribution After Missing Value Treatment')
plt.xticks(rotation=45)
plt.show()

## 4. Train-test split and standardization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

## 5. Train models before hyperparameter tuning

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(X_test)
    else:
        y_score = y_pred

    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC AUC': roc_auc_score(y_test, y_score),
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    }

base_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(kernel='linear', probability=True, random_state=RANDOM_STATE)
}

results = []
trained_base_models = {}

for name, clf in base_models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', clf)
    ])
    pipe.fit(X_train, y_train)
    trained_base_models[name] = pipe
    results.append(evaluate_model(name, pipe, X_test, y_test))

base_results = pd.DataFrame(results).drop(columns=['Confusion Matrix'])
base_results.sort_values('Accuracy', ascending=False)

In [ ]:
base_results.set_index('Model')[['Accuracy','Precision','Recall','F1 Score']].plot(kind='bar', figsize=(10,5))
plt.title('Model Performance Before Hyperparameter Tuning')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.show()

## 6. Hyperparameter tuning

In [ ]:
param_grids = {
    'Decision Tree': {
        'model__max_depth': [3, 5, 7, 10, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [5, 10, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    },
    'Naive Bayes': {
        'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    },
    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },
    'SVM': {
        'model__C': [0.1, 1, 10, 100],
        'model__kernel': ['linear', 'rbf'],
        'model__gamma': ['scale', 'auto']
    }
}

tuned_results = []
best_models = {}

for name, clf in base_models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', clf)
    ])
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    tuned_results.append(evaluate_model(name, grid.best_estimator_, X_test, y_test))
    print(f"{name} best parameters: {grid.best_params_}")

tuned_results_df = pd.DataFrame(tuned_results).drop(columns=['Confusion Matrix'])
tuned_results_df.sort_values('Accuracy', ascending=False)

In [ ]:
tuned_results_df.set_index('Model')[['Accuracy','Precision','Recall','F1 Score']].plot(kind='bar', figsize=(10,5))
plt.title('Model Performance After Hyperparameter Tuning')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.show()

## 7. Best model and detailed report

In [ ]:
best_row = tuned_results_df.sort_values('Accuracy', ascending=False).iloc[0]
best_model_name = best_row['Model']
best_model = best_models[best_model_name]

print('Best Model:', best_model_name)
print(best_row)

y_pred = best_model.predict(X_test)
print('
Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## 8. Predict diabetes for one person

Enter values in this order:

`Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age`

In [ ]:
input_data = (5, 166, 72, 19, 175, 25.8, 0.587, 61)

input_df = pd.DataFrame([input_data], columns=X.columns)
prediction = best_model.predict(input_df)[0]

if prediction == 0:
    print('The person is predicted as Non-Diabetic')
else:
    print('The person is predicted as Diabetic')

## 9. Save best model for future use

In [ ]:
import joblib

joblib.dump(best_model, 'diabetes_best_model.pkl')
print('Model saved as diabetes_best_model.pkl')